# 01 — Cohort & Label Definition

**Project:** 30-Day Readmission Risk in Diabetic Inpatients
**Notebook 1 of 5** — defining who is in the analysis and what we are predicting.

---

## Why this notebook exists

Before any modelling, two questions must be answered and *justified*:

1. **Who counts as a patient in this analysis?** (cohort definition)
2. **What exactly are we predicting?** (label definition)

Most published work on this dataset skips straight to modelling. That is why
much of it is wrong. A model trained on a badly defined cohort produces a
number that looks fine and means nothing.

Every exclusion below is a **clinical** decision, not a data-cleaning one, and
each is justified in writing.

---
## 0. Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

DATA_DIR = "../data"   # gitignored — see data/README.md for download instructions

---
## 1. Load the data

Two files matter:

- `diabetic_data.csv` — one row per hospital **encounter**
- `IDS_mapping.csv` — decodes the numeric IDs for admission type, discharge
  disposition and admission source

Note `na_values="?"`. In this dataset missingness is encoded as a literal
question mark, not as an empty cell. Loading without this silently treats
missing values as a valid category.

In [2]:
df = pd.read_csv(f"{DATA_DIR}/diabetic_data.csv", na_values="?", low_memory=False)

print("Rows (encounters):", f"{len(df):,}")
print("Columns:", df.shape[1])
print("Unique patients:", f"{df['patient_nbr'].nunique():,}")
print("Encounters per patient (mean):",
      round(len(df) / df['patient_nbr'].nunique(), 2))

df.head()

Rows (encounters): 101,766
Columns: 50
Unique patients: 71,518
Encounters per patient (mean): 1.42


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:
df.shape

(101766, 50)

In [4]:
# Track every exclusion so we can report an audit trail in the README.
audit = [("Raw dataset", len(df))]

def log(step, frame):
    audit.append((step, len(frame)))
    print(f"{step:<45} n = {len(frame):>7,}")

log("Raw dataset", df)

Raw dataset                                   n = 101,766


---
## 2. Decode the ID mappings

`IDS_mapping.csv` is stored as three stacked tables in one file, separated by
blank rows. We parse it into three dictionaries.

**This step is where the clinical work starts.** The discharge disposition
codes are the difference between a defensible cohort and a broken one.

In [5]:
# Read as raw text, not with read_csv: pandas skips blank lines by default,
# which destroys the very separator we need to split the three tables on.
with open(f"{DATA_DIR}/IDS_mapping.csv") as f:
    raw_lines = [ln.rstrip("\n").rstrip("\r") for ln in f]

blocks, current = [], []
for ln in raw_lines:
    if not ln.strip() or ln.strip() == ",":     # blank row = table separator
        if current:
            blocks.append(current)
            current = []
    else:
        current.append(ln)
if current:
    blocks.append(current)

print(f"Parsed {len(blocks)} mapping tables")
for b in blocks:
    print(f"  {b[0]:<35} ({len(b) - 1} codes)")

assert len(blocks) == 3, f"Expected 3 tables, got {len(blocks)} - inspect the file"

Parsed 3 mapping tables
  admission_type_id,description       (8 codes)
  discharge_disposition_id,description (30 codes)
  admission_source_id,description     (25 codes)


In [6]:
def to_dict(block):
    """block[0] is the header row; each later row is 'id,description'."""
    out = {}
    for ln in block[1:]:
        code_str, _, desc = ln.partition(",")
        code_str = code_str.strip()
        if code_str.isdigit():
            out[int(code_str)] = desc.strip().strip('"')
    return out

admission_type   = to_dict(blocks[0])
discharge_disp   = to_dict(blocks[1])
admission_source = to_dict(blocks[2])

print("DISCHARGE DISPOSITION CODES")
print("-" * 70)
for k, v in sorted(discharge_disp.items()):
    n = (df["discharge_disposition_id"] == k).sum()
    print(f"{k:>3}  {str(v)[:55]:<57} n={n:>6,}")

DISCHARGE DISPOSITION CODES
----------------------------------------------------------------------
  1  Discharged to home                                        n=60,234
  2  Discharged/transferred to another short term hospital     n= 2,128
  3  Discharged/transferred to SNF                             n=13,954
  4  Discharged/transferred to ICF                             n=   815
  5  Discharged/transferred to another type of inpatient car   n= 1,184
  6  Discharged/transferred to home with home health service   n=12,902
  7  Left AMA                                                  n=   623
  8  Discharged/transferred to home under care of Home IV pr   n=   108
  9  Admitted as an inpatient to this hospital                 n=    21
 10  Neonate discharged to another hospital for neonatal aft   n=     6
 11  Expired                                                   n= 1,642
 12  Still patient or expected to return for outpatient serv   n=     3
 13  Hospice / home                  

In [7]:
rates = (df.groupby("discharge_disposition_id")
           .agg(n=("readmitted", "size"),
                readmit_rate=("readmitted", lambda s: (s == "<30").mean()))
           .sort_values("n", ascending=False))
rates["label"] = rates.index.map(discharge_disp)
print(rates.head(15).to_string())

                              n  readmit_rate                                                                                                      label
discharge_disposition_id                                                                                                                                
1                         60234      0.093004                                                                                         Discharged to home
3                         13954      0.146625                                                                              Discharged/transferred to SNF
6                         12902      0.126957                                                    Discharged/transferred to home with home health service
18                         3691      0.124357                                                                                                       NULL
2                          2128      0.160714                                     

## Reasoning: discharge disposition

Discharge disposition is the most clinically loaded variable in this dataset, and
the exclusions below are made for **four different reasons**, not one. Grouping
them under a single "data cleaning" heading would hide the actual arguments.

**1. Deceased — codes 11, 19, 20, 21. Excluded: label leakage.**

These patients died. Every one is a guaranteed `readmitted = NO`, so the model
finds a perfect predictor. The problem is not only that AUC inflates on an unearned
separation — it is that discharge disposition is recorded *at discharge*, which is
the exact moment the prediction is needed. A model that scores a patient as low
risk because they died has answered a question nobody asked, and the apparent
performance gain disappears the moment it is deployed.

**2. Hospice — codes 13, 14. Excluded: wrong target population.**

This is not leakage. A hospice patient can genuinely be readmitted, and the data
would record it correctly. The failure is at the point of use. If the model flags a
hospice patient as high risk, the system recommends medication reconciliation, a
follow-up appointment and a 72-hour call — interventions that are clinically
inappropriate for a patient whose care goal is comfort rather than readmission
avoidance. Because a discharge team can only work through a limited list each
morning, that patient also occupies a slot that should have gone to someone the
intervention could help. The model would work; the deployment would be wrong.

**3. Encounter never ended — codes 9, 12. Excluded: index event undefined.**

Code 9 records the patient as admitted as an inpatient to this same facility, and
code 12 as still a patient or expected to return. In both cases the index encounter
has not concluded, so the 30-day clock has no start point. These labels are not
incorrect — they are undefined.

**4. Destination not recorded — codes 18, 25, 26. Retained as an explicit
"unknown" category.**

These encounters have no usable destination recorded. The patient was discharged
and their readmission outcome is observed normally, so there is no censoring
problem here — this is missing data on the *predictor* side, not the outcome side.

They are retained, but on one condition that must hold at deployment: `unknown` is
treated as its own category the model is trained on, and any live system must be
able to emit that same category when the field is blank. This matters because a
feature can be perfectly informative during training and simply unavailable at
prediction time — one of the commonest reasons clinical models fail between the
notebook and the ward. Code 18 carries a 12.4% readmission rate against 9.3% for
routine home discharge, so "no destination recorded" may itself be a marker of a
disorganised or hurried discharge. That is plausible signal, but it is a
data-quality artefact being modelled as a clinical feature, and it is recorded here
as such.

**Retained: all transfer destinations, and Left AMA (code 7).**

Transfers were initially considered for exclusion on the assumption that a
readmission occurring outside this 130-hospital network would be recorded as `NO`,
making the negative class unreliable. That assumption was tested rather than
accepted (see the readmission-rate-by-disposition analysis above). Observed rates
rise monotonically with destination acuity — home 9.3%, home health 12.7%, ICF
12.8%, SNF 14.7%, another short-term hospital 16.1%, other inpatient institution
20.9%, rehabilitation 27.7%. Under systematic under-capture these sicker groups
would show artificially *lower* rates than home discharge. They do not, so the
transfer codes are retained.

This test is one-directional and is reported as such: a low rate would have been
strong evidence of under-capture, but a high rate does not prove complete capture,
since transferred patients are genuinely sicker and could exceed the home-discharge
rate even with some readmissions missed. The conclusion is therefore *no evidence
of severe under-capture*, not *proof of full ascertainment*.

One anomaly is noted rather than acted on. Long-term care hospital (code 23) shows
7.3% against n=412 — below home discharge, in a population that should be sicker.
The likely explanation is competing exposure time rather than under-capture: LTCH
stays routinely exceed 30 days, so a patient still resident at day 30 was never at
risk of an acute readmission during the window. With roughly 30 events the estimate
is imprecise, and the effect is not large enough to justify an exclusion.

Left AMA (code 7) is retained. Self-discharge means treatment was incomplete by
definition, so these patients carry elevated risk — but they went home locally and
their outcome is observed normally. High risk is the signal the model exists to
detect, not a reason for exclusion. Worth noting for interpretation: AMA is largely
a socioeconomic marker wearing clinical clothing. It reflects cost pressure,
caregiving obligations, transport difficulty or a broken relationship with the
health system — none of which this dataset measures directly. If the model finds it
predictive, the mechanism is not that self-discharge causes readmission, but that
both are downstream of circumstances never recorded.ify an exclusion.harge causes
readmission, but that both are downstream of circumstances never recorded.

---
## 3. Applying the discharge-disposition exclusions

Three groups:

- **Deceased** — expired in hospital or shortly after discharge
- **Hospice** — discharged to comfort-focused care; a readmission is not the
  outcome we are trying to prevent, and preventing it is not the clinical goal
- **Ambiguous / not evaluable** — dispositions where follow-up is undefined

Adjust the code list below to match the reasoning you wrote above. The numbers
here are a starting point, not gospel — **check them against the printed
mapping and change them if you disagree.**

In [8]:
# Deaths — outcome impossible; retaining them is label leakage.
EXCLUDE_DECEASED = [11, 19, 20, 21]
# Hospice — outcome possible, but wrong target population for this intervention.
EXCLUDE_HOSPICE = [13, 14]
# Encounter never ended — no discharge, so the 30-day clock cannot start.
EXCLUDE_NO_DISCHARGE = [9, 12]

for label, codes in [("Deceased", EXCLUDE_DECEASED),
                     ("Hospice", EXCLUDE_HOSPICE),
                     ("Never discharged", EXCLUDE_NO_DISCHARGE)]:
    print(f"{label}:")
    for c in codes:
        print(f"  {c:>3}  {discharge_disp.get(c)}")

df = df[~df["discharge_disposition_id"].isin(EXCLUDE_DECEASED)]
log("After removing deceased", df)

df = df[~df["discharge_disposition_id"].isin(EXCLUDE_HOSPICE)]
log("After removing hospice discharges", df)

df = df[~df["discharge_disposition_id"].isin(EXCLUDE_NO_DISCHARGE)]
log("After removing encounters that never ended", df)

Deceased:
   11  Expired
   19  Expired at home. Medicaid only, hospice.
   20  Expired in a medical facility. Medicaid only, hospice.
   21  Expired, place unknown. Medicaid only, hospice.
Hospice:
   13  Hospice / home
   14  Hospice / medical facility
Never discharged:
    9  Admitted as an inpatient to this hospital
   12  Still patient or expected to return for outpatient services
After removing deceased                       n = 100,114
After removing hospice discharges             n =  99,343
After removing encounters that never ended    n =  99,319


---
## 4. Repeat encounters from the same patient

`patient_nbr` repeats: some patients appear many times. This creates two
problems.

**Statistical:** encounters from the same patient are correlated. Treating them
as independent observations overstates your effective sample size and inflates
confidence in the result.

**Leakage:** if the same patient's encounters land in both the training and
test sets, the model can memorise that patient rather than learn a pattern.
Your test score is then measuring memorisation, not generalisation. This is the
single most common error in published work on this dataset.

Let us first see how bad it is.

In [9]:
raw = pd.read_csv(f"{DATA_DIR}/diabetic_data.csv", na_values="?", low_memory=False)
raw = raw[~raw["discharge_disposition_id"].isin(
    EXCLUDE_DECEASED + EXCLUDE_HOSPICE + EXCLUDE_NO_DISCHARGE)]

c = raw["patient_nbr"].value_counts()
print("Encounters:", f"{len(raw):,}")
print("Unique patients:", f"{raw['patient_nbr'].nunique():,}")
print("Patients with >1 encounter:", f"{(c > 1).sum():,}")
print("Max encounters, one patient:", c.max())
print()
print(c.value_counts().sort_index().head(10).to_string())
print()

raw["y"] = (raw["readmitted"] == "<30").astype(int)
raw["n_enc"] = raw["patient_nbr"].map(c)
print(raw.groupby(raw["n_enc"] > 1)["y"].agg(["size", "mean"]).to_string())

Encounters: 99,319
Unique patients: 69,980
Patients with >1 encounter: 16,337
Max encounters, one patient: 40

count
1     53643
2     10216
3      3232
4      1355
5       689
6       339
7       191
8       113
9        66
10       41

        size      mean
n_enc                 
False  53643  0.042634
True   45676  0.197390


In [10]:
# readmission rate for repeat vs single-encounter patients:

raw["target"] = (raw["readmitted"] == "<30").astype(int)
raw["n_enc"] = raw["patient_nbr"].map(raw["patient_nbr"].value_counts())
print(raw.groupby(raw["n_enc"] > 1)["target"].agg(["size", "mean"]).to_string())

        size      mean
n_enc                 
False  53643  0.042634
True   45676  0.197390


## Repeat encounters: handled at the split, not by dropping rows

The same patient shows up more than once in this data. After the discharge
exclusions above I'm left with 99,319 encounters from 69,980 patients. About a
quarter of those patients (16,337) came back at least once, and one came back 40
times. Between them, repeat patients account for 45,676 rows, so nearly half the
dataset is people who have been admitted before.

That's a leakage problem. If the same patient lands in both my training and test
sets, the model can just recognise them instead of learning anything general, and
my test score stops meaning what I think it means.

### The three options

**(a) Keep only each patient's first encounter.** Simple, and it guarantees no
patient can straddle the split. But it throws away every subsequent admission.

**(b) Keep all encounters, but split train/test by patient** so a patient appears
in one side or the other, never both. Keeps the data, moves the fix to the split.

**(c) Keep all encounters and split randomly**, accepting the correlation and
documenting it. Cheapest to implement and the least defensible — the same patient
ends up in train and test and the model memorises them.

(c) is not really an option; it's the error the other two exist to prevent. So
the real choice was between (a) and (b), and I looked at what (a) would actually
cost before deciding.

### Why I ruled out (a)

First I checked whether repeat patients even differ:

| Group | Encounters | 30-day readmission rate |
|---|---|---|
| Single-encounter patients | 53,643 | 4.3% |
| Repeat patients | 45,676 | 19.7% |

They differ a lot. Nearly five times the readmission rate.

So keeping first encounters only would mean throwing away 29,343 rows, and almost
all of them sit in the group where the events are. That's roughly 5,800 positive
cases gone from a problem that doesn't have many to begin with.

It would also quietly move the base rate from about 11% to about 8%, which
matters because every threshold I pick later depends on it.

The part that bothered me most is what it does to the feature set. If every
patient has exactly one admission, then prior admission history can't exist as a
variable. And prior admissions are the first thing I'd look at on a ward round.
A model running in a real hospital is mostly scoring people who have been in
before. Training it on a population where nobody has been in before means it has
never seen the patients it will actually be used on.

### What I did instead — option (b)

I kept all 99,319 encounters and dealt with the leakage where it actually
happens, at the split. In notebook 03 I use `GroupShuffleSplit` grouped on
`patient_nbr`, so a patient can be in training or test but never both.

Same protection, without paying 30% of the data for it.

Worth being clear that this isn't an exclusion at all — no rows are dropped, so
there's no Exclusion 2 row in the audit trail. It's a train/test design decision,
and it belongs in the methodology rather than the cohort definition.

I'm predicting encounters, not patients, which fits the clinical question:
this patient, being discharged today, will they be back within 30 days?

### What this costs me

**The rows aren't independent.** Encounters from the same patient are related, so
my effective sample size is smaller than 99,319 looks. Confidence intervals
calculated as if every row were independent will be too narrow. I need to account
for clustering later, or say clearly that I haven't.

**The 4.6× gap isn't purely clinical.** Frequent admitters really are sicker, and
that's most of it. But someone with more admissions also has more chances for two
of them to land within 30 days of each other, so some of that gap is just
arithmetic. I can't separate the two with this data, and I shouldn't quote the
number as if it were a clean clinical effect.

**One thing that surprised me.** Single-encounter patients still show a 4.3%
readmission rate. If `readmitted` were derived only from the encounters in this
file, that should be zero — the readmission would itself appear as a second row
for that patient, and these patients have only one row. Since it isn't zero, the
outcome must have been recorded from somewhere else, most likely claims data.
Worth confirming in the dataset documentation, but if true it's reassuring:
outcomes are being **A note to myself for notebook 03.** Prior encounter count will probably be a
strong predictor, but I have to count only admissions that happened *before* the
one I'm scoring. Counting the whole record includes the future, which is leakage.fore* the
one I'm scoring. Counting the whole record includes the future, which is leakage. codes.* the
one I'm scoring. Counting the whole record includes the future, which is leakage.total count across the whole
record includes future encounters and is leakage.

In [11]:
c = df["patient_nbr"].value_counts()
print(f"Encounters:              {len(df):,}")
print(f"Unique patients:         {df['patient_nbr'].nunique():,}")
print(f"Patients with >1 encounter: {(c > 1).sum():,}")
print(f"Max encounters for one patient: {c.max()}")
print("\nAll encounters retained. Patient-level grouping is applied at the")
print("train/test split in notebook 03 (GroupShuffleSplit on patient_nbr).")

Encounters:              99,319
Unique patients:         69,980
Patients with >1 encounter: 16,337
Max encounters for one patient: 40

All encounters retained. Patient-level grouping is applied at the
train/test split in notebook 03 (GroupShuffleSplit on patient_nbr).


---
## 5. Clinically implausible records


In [12]:
print("GENDER")
print(df["gender"].value_counts(dropna=False).to_string())
print()
print("ADMISSION TYPE")
for k, v in sorted(admission_type.items()):
    n = (df["admission_type_id"] == k).sum()
    if n:
        print(f"{k:>3}  {str(v)[:40]:<42} n={n:>6,}")
print()
print("AGE BANDS")
print(df["age"].value_counts().sort_index().to_string())

GENDER
gender
Female             53444
Male               45872
Unknown/Invalid        3

ADMISSION TYPE
  1  Emergency                                  n=52,360
  2  Urgent                                     n=18,126
  3  Elective                                   n=18,666
  4  Newborn                                    n=    10
  5  Not Available                              n= 4,614
  6  NULL                                       n= 5,205
  7  Trauma Center                              n=    18
  8  Not Mapped                                 n=   320

AGE BANDS
age
[0-10)        160
[10-20)       689
[20-30)      1649
[30-40)      3764
[40-50)      9605
[50-60)     17060
[60-70)     22048
[70-80)     25326
[80-90)     16430
[90-100)     2588


In [13]:
# 'Unknown/Invalid' gender — a handful of records with no recoverable value.
df = df[df["gender"] != "Unknown/Invalid"]
log("After removing unknown/invalid gender", df)

# Newborn admissions (type 4) — inspect before deciding.
n_newborn = (df["admission_type_id"] == 4).sum()
print(f"\nNewborn admissions present: {n_newborn}")

After removing unknown/invalid gender         n =  99,316

Newborn admissions present: 10


In [14]:
# 'Newborn' inspection

nb = df[df["admission_type_id"] == 4]
print("admission_type_id == 4 (labelled 'Newborn'):", len(nb))
print()
print(nb[["age", "diag_1", "discharge_disposition_id"]].head(10).to_string())

admission_type_id == 4 (labelled 'Newborn'): 10

            age  diag_1  discharge_disposition_id
2043    [50-60)     414                         1
2203    [80-90)     414                         1
2461    [70-80)     562                         6
4823    [60-70)  250.82                         6
35877   [60-70)     276                         1
47548   [40-50)     870                         1
48711   [70-80)     715                         6
80354    [0-10)     786                         1
87714   [60-70)     435                         1
100721  [80-90)     491                         6


In [15]:
d10 = df[df["discharge_disposition_id"] == 10]
print("discharge_disposition_id == 10 (labelled 'neonate'):", len(d10))
print()
print(d10[["admission_type_id", "age", "diag_1"]].head(10).to_string())

discharge_disposition_id == 10 (labelled 'neonate'): 6

      admission_type_id       age diag_1
487                   6   [70-80)    434
1027                  6   [70-80)    434
1101                  6   [60-70)    V57
1585                  6  [90-100)    820
1636                  6   [80-90)    715
1983                  6   [50-60)    715


In [16]:
# 'Paediatric' inspection

peds = df[df["age"] == "[0-10)"]
print("Rows in [0-10):", len(peds))

def perinatal(s):
    return s.astype(str).str.match(r"^7[67][0-9]")

for col in ["diag_1", "diag_2", "diag_3"]:
    print(f"{col}: {perinatal(peds[col]).sum()} perinatal codes (760-779)")

mask = (df["diag_1"].astype(str).str.startswith("775") |
        df["diag_2"].astype(str).str.startswith("775") |
        df["diag_3"].astype(str).str.startswith("775"))
print("\nInfant of diabetic mother (775.x) anywhere:", mask.sum())

print("\nAdmission source for [0-10):")
print(peds["admission_source_id"].value_counts().to_string())

Rows in [0-10): 160
diag_1: 0 perinatal codes (760-779)
diag_2: 0 perinatal codes (760-779)
diag_3: 0 perinatal codes (760-779)

Infant of diabetic mother (775.x) anywhere: 0

Admission source for [0-10):
admission_source_id
7     108
1      38
4      10
17      4


In [17]:
# Profiling the paediatric group 

peds_all = df[df["age"].isin(["[0-10)", "[10-20)"])]
print("Under 20:", len(peds_all))
print(peds_all["age"].value_counts().to_string())

events = (peds_all["readmitted"] == "<30").sum()
print(f"\nEvents: {events}")
print(f"Readmission rate: {events / len(peds_all):.1%}")
print(f"Cohort rate:      {(df['readmitted'] == '<30').mean():.1%}")

print("\nTop diagnoses:")
print(peds_all["diag_1"].value_counts().head(10).to_string())

Under 20: 849
age
[10-20)    689
[0-10)     160

Events: 43
Readmission rate: 5.1%
Cohort rate:      11.4%

Top diagnoses:
diag_1
250.13    237
250.11    129
250.03    101
250.01     38
250.02     31
482        22
648        15
296        14
682        14
250.83     13


## Implausible records: what I checked and what I found

The scaffold for this section asked whether newborn admissions belong in a
diabetic inpatient cohort. Before answering that I checked whether there are any,
and the answer changed the question.

### There are no newborns

`admission_type_id == 4` is labelled "Newborn". It has 10 rows. Nine of them are
patients aged 40 to 90, with coronary atherosclerosis, diverticular disease,
chronic bronchitis, osteoarthritis and a superficial injury. The tenth is in the
`[0-10)` band with a respiratory symptom code. None of them is a newborn.

I checked properly rather than relying on the field label:

- No perinatal codes (ICD-9 760–779) in `diag_1`, `diag_2` or `diag_3` for any
  patient in the `[0-10)` band
- No code 775.x (infant of a diabetic mother) anywhere in the 99,316 encounters
- No birth-related admission sources (11–14) in the paediatric band — the sources
  present are emergency room, physician referral and hospital transfer

So there is nothing to exclude here. This also makes sense given the dataset's own
inclusion criteria: encounters had to be inpatient stays of 1–14 days with
laboratory tests performed and medications administered, which routine newborn
admissions would not meet.

### Two mislabelled columns

While checking, I found the same problem twice.

`admission_type_id == 4` says Newborn and contains adults.
`discharge_disposition_id == 10` says "transferred to another short-term hospital,
neonate" and contains patients aged 50 to 100 with stroke, hip fracture,
rehabilitation aftercare and osteoarthritis.

Both are coding errors in the source data rather than populations to remove — the
patients are real and what happened to them is unambiguous, only the label is
wrong. I have left them in. Code 10 is a transfer and is covered by the transfer
analysis above.

Worth recording as a limitation: two field labels in a widely used public dataset
do not describe their contents. Any feature engineering that trusts these labels
without inspecting the rows will encode the error.

### The paediatric group — considered for exclusion, kept

The `[0-10)` and `[10-20)` bands together hold 849 encounters (0.9% of the cohort)
with 43 readmission events.

The diagnoses are strikingly homogeneous. 549 of the 849 encounters carry a type 1
diabetes code, and 366 of those are type 1 with ketoacidosis. This is not a scatter
of incidental paediatric admissions — it is children and adolescents admitted in
DKA. A further 15 encounters are coded 648, diabetes complicating pregnancy, in the
`[10-20)` band.

I considered excluding them. The argument for doing so is real: a 7-year-old in DKA
and a 74-year-old with type 2 diabetes and renal impairment share a diagnosis label
and very little else. The drivers of readmission differ — insulin omission, school
and family circumstances, adolescent adherence, against polypharmacy, renal
function and frailty. So does the appropriate intervention. A model fit
overwhelmingly on adult type 2 patients is not obviously fit to score them.

The data supports that concern. The under-20 readmission rate is 5.1% against 11.4%
for the cohort overall — less than half. An adult-fit model should be expected to
systematically over-predict risk in this group.

I kept them anyway, for two reasons. First, they are genuine diabetic inpatient
admissions with genuine readmission outcomes; excluding real patients on an
assumption about model behaviour is weaker than measuring that behaviour. Second,
43 events is enough to detect a difference of the size observed, so the concern is
testable rather than hypothetical.

The exclusion is therefore replaced by three commitments carried into evaluation:

1. `[0-10)` and `[10-20)` are reported as a single under-20 subgroup in notebook 04.
   Separately they are too sparse; combined they are clinically coherent, given the
   diagnosis profile.
2. **Calibration is reported for this subgroup, not only discrimination.** With a
   base rate less than half the cohort average, the likely failure mode is correct
   rank-ordering with wrong absolute risk — which discrimination metrics would
   conceal entirely.
3. The intended-use statement records that the model is developed on a
   predominantly adult, predominantly type 2 population; that under-20s comprise
   0.9% of the cohort and are predominantly type 1 presenting in DKA; and that any
   paediatric performance estimate rests on 43 events.

This is stratified evaluation rather than stratified modelling. One model, with
subgroup performance reported and confidence intervals shown, so that the
imprecision is visible rather than implied.
imprecision is visible rather than implied.and confidence intervals shown, so that the
imprecision is visible rather than implied.

In [18]:
# check all variables in the dataset

for c in df.columns:
    vc = df[c].value_counts(dropna=False)
    print(f"{c:<28} miss:{df[c].isna().mean():>5.1%}  uniq:{df[c].nunique():>4}  "
          f"top:{str(vc.index[0])[:18]:<20} {vc.iloc[0]/len(df):>5.1%}")

encounter_id                 miss: 0.0%  uniq:99316  top:2278392               0.0%
patient_nbr                  miss: 0.0%  uniq:69977  top:88785891              0.0%
race                         miss: 2.2%  uniq:   5  top:Caucasian            74.7%
gender                       miss: 0.0%  uniq:   2  top:Female               53.8%
age                          miss: 0.0%  uniq:  10  top:[70-80)              25.5%
weight                       miss:96.9%  uniq:   9  top:nan                  96.9%
admission_type_id            miss: 0.0%  uniq:   8  top:1                    52.7%
discharge_disposition_id     miss: 0.0%  uniq:  19  top:1                    60.6%
admission_source_id          miss: 0.0%  uniq:  17  top:7                    56.2%
time_in_hospital             miss: 0.0%  uniq:  14  top:3                    17.6%
payer_code                   miss:39.7%  uniq:  17  top:nan                  39.7%
medical_specialty            miss:48.9%  uniq:  72  top:nan                  48.9%
nu

In [19]:
drugs = [c for c in df.columns if df[c].dtype == object and
         set(df[c].dropna().unique()) <= {"No", "Steady", "Up", "Down"}]

for c in drugs:
    vc = df[c].value_counts()
    if vc.iloc[0] / len(df) > 0.99:
        print(f"{c:<28} {dict(vc)}")

nateglinide                  {'No': 98627, 'Steady': 654, 'Up': 24, 'Down': 11}
chlorpropamide               {'No': 99231, 'Steady': 78, 'Up': 6, 'Down': 1}
acetohexamide                {'No': 99315, 'Steady': 1}
tolbutamide                  {'No': 99295, 'Steady': 21}
acarbose                     {'No': 99011, 'Steady': 292, 'Up': 10, 'Down': 3}
miglitol                     {'No': 99278, 'Steady': 31, 'Down': 5, 'Up': 2}
troglitazone                 {'No': 99313, 'Steady': 3}
tolazamide                   {'No': 99277, 'Steady': 38, 'Up': 1}
examide                      {'No': 99316}
citoglipton                  {'No': 99316}
glyburide-metformin          {'No': 98618, 'Steady': 684, 'Up': 8, 'Down': 6}
glipizide-metformin          {'No': 99303, 'Steady': 13}
glimepiride-pioglitazone     {'No': 99315, 'Steady': 1}
metformin-rosiglitazone      {'No': 99314, 'Steady': 2}
metformin-pioglitazone       {'No': 99315, 'Steady': 1}


In [20]:
freq = df["patient_nbr"].value_counts()
sample = freq[freq >= 4].index[:3]

print(df[df["patient_nbr"].isin(sample)]
        .sort_values(["patient_nbr", "encounter_id"])
        [["patient_nbr", "encounter_id", "number_inpatient",
          "number_emergency", "number_outpatient"]]
        .head(30).to_string())

       patient_nbr  encounter_id  number_inpatient  number_emergency  number_outpatient
269        1660293       2967810                 0                 0                  0
944        1660293       7216812                 1                 0                  0
1077       1660293       7977342                 2                 0                  0
1349       1660293       9358128                 3                 0                  0
2954       1660293      18844260                 4                 0                  0
3510       1660293      21616398                 5                 0                  0
6780       1660293      33120744                 5                 0                  0
7704       1660293      36030504                 6                 0                  0
8547       1660293      38486058                 5                 0                  0
8972       1660293      39749172                 6                 0                  0
12206      1660293      49882548

In [21]:
first = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")
print("number_inpatient at each patient's first encounter in this file:")
print(first["number_inpatient"].value_counts().sort_index().head(8).to_string())

number_inpatient at each patient's first encounter in this file:
number_inpatient
0    61782
1     5798
2     1501
3      463
4      228
5      102
6       55
7       19


In [22]:
tmp = df.assign(y=(df["readmitted"] == "<30").astype(int))
print(tmp.groupby("number_inpatient")["y"]
         .agg(["size", "mean"]).head(12).to_string())

                   size      mean
number_inpatient                 
0                 66230  0.085867
1                 18980  0.132350
2                  7297  0.179252
3                  3270  0.210092
4                  1573  0.241577
5                   790  0.320253
6                   474  0.348101
7                   261  0.360153
8                   145  0.462069
9                   108  0.425926
10                   59  0.440678
11                   49  0.673469


In [23]:
DROP_CONSTANT = ["examide", "citoglipton", "acetohexamide", "tolbutamide",
                 "miglitol", "troglitazone", "tolazamide",
                 "glipizide-metformin", "glimepiride-pioglitazone",
                 "metformin-rosiglitazone", "metformin-pioglitazone"]
DROP_MISSING  = ["weight"]

to_drop = [c for c in DROP_CONSTANT + DROP_MISSING if c in df.columns]
df = df.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} columns:")
for c in to_drop:
    print(" -", c)
print(f"\nRemaining: {df.shape[1]} columns, {len(df):,} encounters")

Dropped 12 columns:
 - examide
 - citoglipton
 - acetohexamide
 - tolbutamide
 - miglitol
 - troglitazone
 - tolazamide
 - glipizide-metformin
 - glimepiride-pioglitazone
 - metformin-rosiglitazone
 - metformin-pioglitazone
 - weight

Remaining: 38 columns, 99,316 encounters


## Variable audit: what survives into modelling

Before moving to the label, I audited all 50 columns for three failure modes:
information that would not exist at the moment of prediction, values that make no
clinical sense, and variables that do not vary.

### Dropped — no information

`examide` and `citoglipton` have a single unique value across all 99,316 rows.
Every patient is recorded as No.

A second tier is technically non-constant but effectively so. `acetohexamide`,
`tolbutamide`, `miglitol`, `troglitazone`, `tolazamide`, `glipizide-metformin`,
`glimepiride-pioglitazone`, `metformin-rosiglitazone` and `metformin-pioglitazone`
each have fewer than 40 patients on any active regimen out of 99,316. These are
older or unusual agents. They cannot support a stable estimate and they inflate the
feature count without adding signal.

Four others sit above that threshold but are still heavily skewed — `nateglinide`,
`chlorpropamide`, `acarbose` and `glyburide-metformin`, each with between 80 and
700 patients on an active regimen. These are retained: the numbers are small but
large enough that a tree-based model could find a real split, and unlike the
dropped group they represent agents in current clinical use.

### Dropped — unusable missingness

`weight` is 96.9% missing.

This is a genuine loss rather than a nuisance. Body habitus is a real predictor of
diabetic outcome and disease progression, and overweight and obese patients do
worse. But 3% coverage cannot be imputed honestly, and the 3% who have a recorded
weight are very unlikely to be a random sample of the cohort. It is recorded in the
limitations as a substantive gap in the available predictors.

### Kept with missingness treated explicitly

`payer_code` (39.7% missing) is retained. It is an insurance proxy, which makes it
a socioeconomic variable in clinical clothing — much like discharge against medical
advice. It plausibly predicts readmission, but the mechanism would be access to
follow-up care rather than anything physiological. Two consequences follow: the
missingness is unlikely to be random, and using it means scoring patients partly on
their insurance status. That is a fairness question, and it is carried into
notebook 04 rather than settled silently here.

`medical_specialty` (48.9% missing) is retained with missing as an explicit
category. The admitting service is clinically meaningful — a patient on a
cardiology service is a different proposition from one on general medicine — and it
is known at discharge.

`race` (2.2% missing) is retained. It is required for the subgroup performance
analysis, which is the point of collecting it.

### The laboratory results are not missing data

`A1Cresult` is 83.1% blank and `max_glu_serum` is 94.8% blank. Read as missing
values these look unusable. I don't think that's the right reading.

A blank `A1Cresult` most likely means the test was not ordered, rather than that a
result exists somewhere and wasn't transferred — the original study on this dataset
framed the variable as measurement status. I have not confirmed this from the
documentation directly, so I state it as an assumption rather than as established
fact. If it is wrong, the measured flag is a data-completeness artefact rather than
a clinical process variable, and any predictive signal it carries would need
re-interpreting.

If the assumption holds, the blank is a recorded fact about the admission, fully
known at discharge, and may be more informative than the result itself. Clinically,
a diabetic inpatient whose HbA1c was never checked describes an admission where
diabetes was not the focus, or the workup was thin, or the team was stretched. That
is plausibly a marker of discharge quality, which is precisely what predicts
returning. It is also the question the original study on this dataset asked —
whether HbA1c *measurement* was associated with readmission.

Both variables are therefore split into two features in notebook 03: a binary
measured flag, and the result category with `not_measured` as an explicit level.

Two caveats. This makes the flag a process variable, not a physiological one — if
it predicts, the mechanism concerns how the admission was managed, not glycaemic
control, and it must not be read as "HbA1c predicts readmission". And among the 17%
with a result, the population is selected: clinicians order the test when they
suspect poor control, so measured and unmeasured patients are not comparable. That
is confounding by indication and belongs in the limitations.

### Prior utilisation: tested for leakage, and clean

`number_inpatient`, `number_emergency` and `number_outpatient` are documented as
visit counts for the year preceding the encounter. If that were wrong — if they
included the index encounter or later ones — they would be leakage in its purest
form, using tomorrow's admission to predict tomorrow's admission.

Having already found two mislabelled columns in this dataset, I tested rather than
trusted the documentation.

**Within-patient trajectory.** Following individual patients across successive
encounters in `encounter_id` order, `number_inpatient` increments as admissions
accumulate — 0, 1, 2, 3, 4, 5 for one patient's first six encounters. A fixed
per-patient total would show the same value on every row. It does not. The counts
are historical.

The counts also *decrease* at points — one patient runs 9, 7, 7, 7; another falls
from 13 to 12 to 11. A cumulative lifetime count cannot decrease. This confirms a
rolling 12-month window, with older admissions ageing out as new ones enter, which
matches the documented definition and is the more useful behaviour: recent
utilisation predicts near-term readmission better than a lifetime total.

**Dose-response.** Readmission rate rises monotonically with prior inpatient visits
— 8.6% at zero, 13.2% at one, 17.9% at two, 21.0% at three, 24.2% at four, 32.0% at
five, 34.8% at six, 46.2% at eight. A smooth five-fold gradient, and clinically
exactly what would be expected.

Equally important, it is not *too* strong. Leakage would look like near-zero risk at
zero prior visits and near-certainty above three. Even patients with eight prior
admissions remain below 50%. The variability at nine and above reflects small
numbers (108 and 49 encounters respectively) and is not over-interpreted.

All three variables are retained. This also resolves a question left open by the
repeat-encounter decision above: prior admission history is already present as a
correctly constructed feature, so it does not need to be engineered from
`patient_nbr`, and the associated risk of accidentally counting future encounters
does not arise.

### Carried forward

`diag_1`, `diag_2` and `diag_3` contain 715, 747 and 786 distinct ICD-9 codes.
These need grouping into clinical categories before modelling — a notebook 03 task,
not a reason to drop them. need grouping into clinical categories before modelling — a notebook 03
task, not a reason to drop them.

---
## 6. Label definition

`readmitted` has three values:

| Value | Meaning |
|---|---|
| `<30` | readmitted within 30 days |
| `>30` | readmitted, but after 30 days |
| `NO`  | no recorded readmission |


In [24]:
print(df["readmitted"].value_counts(dropna=False).to_string())
print()
print((df["readmitted"].value_counts(normalize=True) * 100).round(1).to_string())

readmitted
NO     52513
>30    35500
<30    11303

readmitted
NO     52.9
>30    35.7
<30    11.4


### MY REASONING — the label

## The label: 30-day readmission

`readmitted` arrives as three categories:

| Value | n | % |
|---|---|---|
| `NO` | 52,513 | 52.9% |
| `>30` | 35,500 | 35.7% |
| `<30` | 11,303 | 11.4% |

I model this as binary — `<30` is the positive class, everything else negative.
That gives a base rate of 11.4%, roughly a 1:8 imbalance, which will drive every
evaluation decision in notebook 03. Accuracy will be useless here.

### What the positive class means

A patient discharged alive from one of these 130 hospitals who was admitted again
as an inpatient within 30 days of that discharge, for any cause.

Three things are packed into that sentence and each is a limitation.

**"For any cause."** The dataset does not distinguish planned from unplanned
readmission, or diabetes-related from unrelated. A patient returning for scheduled
chemotherapy is scored identically to one returning in ketoacidosis. Medicare's
own readmission measure excludes planned admissions; this data does not let me.
Some proportion of my positive class is therefore not a discharge failure at all.

**"As an inpatient."** An emergency department visit that did not result in
admission is not counted, even though clinically it often represents the same
failure — the patient deteriorated after discharge and returned. The label
captures a billing event rather than a clinical one.

**"From one of these 130 hospitals."** The unit is the network, not a single
facility, which is appropriate since the model is being built for the network.

### What the negative class contains

Patients who were never readmitted, and patients who were readmitted after more
than 30 days.

This class is heterogeneous by construction, and it is worth being blunt about
how heterogeneous. **35.7% of the cohort came back — just later.** Clinically, a
patient returning on day 45 has considerably more in common with my positive class
than with someone who never returned at all, yet the label treats the two as
identical.

The consequence: this model learns to separate early returns from everything
else. That is not the same thing as identifying patients who will do badly. A
discharge team using it should understand which of those two questions it
actually answers.

### Is 30 days clinically meaningful?

Not really. It is an administrative boundary that has been widely adopted as
though it were a clinical one.

The window exists because Medicare's Hospital Readmissions Reduction Program
penalises hospitals financially on 30-day readmission rates. That is why the field
is coded this way and why the literature is full of 30-day models. Nothing changes
physiologically between day 29 and day 31.

The usual defence is that early readmissions are more plausibly attributable to
the index admission — incomplete treatment, premature discharge, inadequate
discharge planning — while later ones reflect disease progression. There is
something to that, but nobody argues for a discontinuity at day 30. The
underlying risk is continuous; the line drawn through it is policy.

The clinical reality is that patients come back for a mix of reasons — disease
progression, medication adherence, socioeconomic support, age, the presence of
someone at home to help. None of these has a 30-day structure. That over a third
of this cohort was readmitted beyond the cut-off is itself evidence that the
phenomenon does not stop at the boundary.

### Why I model it anyway

The clinically correct formulation is time-to-event: a continuous hazard of
readmission, modelled with survival analysis, with covariates for progression,
adherence and social circumstances. **This dataset cannot support it.**
`readmitted` is supplied as three categories with no dates attached, so no
time-to-event structure is recoverable. The constraint is in the data, not in the
choice of method.

Within that constraint, binary at 30 days is the right framing for this
particular tool — because the intervention it supports has a shorter horizon than
the risk itself. The discharge-planning actions in question (medication
reconciliation, a diabetes educator review, a follow-up appointment booked before
the patient leaves, a call at 72 hours) act most directly on the weeks
immediately after discharge. Longer-horizon readmissions are shaped by disease
progression, adherence and social circumstances better addressed through ongoing
chronic care than through a discharge checklist. Both matter. They are different
interventions, and this model is built for the first. The prediction window is
matched to the action window rather than to the biology.

Had the use case been "which patients need enhanced chronic disease follow-up",
this decision would have gone the other way.

### Options considered and rejected

**Dropping the `>30` group** would have produced a cleaner contrast and a more
comfortable 17.7% base rate. It was rejected on three grounds: it discards 35.7%
of the data; the resulting base rate is an artefact of deletion rather than the
true rate of 30-day readmission, so every calibrated probability would be
inflated; and a model that has never seen a slow returner will meet them
constantly in deployment.

**A three-class ordinal model** over `NO` → `>30` → `<30` would preserve some of
the timing information a binary cut-off discards, and is the closest available
approximation to survival analysis. It was rejected for now on both clinical and
practical grounds. Clinically, the ordinal assumption is that the three classes
lie on a single underlying severity dimension — and it is not obvious that they
do. A readmission at day 200 driven by progressive nephropathy is arguably a
different outcome from a bounce-back at day 5 in ketoacidosis, not a milder
version of it. Practically, the model must produce a single ranking for a
discharge team to work down, so three probabilities would have to be collapsed
into one anyway, and multiclass calibration is materially harder to compute and
to communicate. It is recorded in future work as a worthwhile extension.

In [25]:
df["target"] = (df["readmitted"] == "<30").astype(int)

rate = df["target"].mean()
print(f"Final cohort:      {len(df):,} encounters")
print(f"Positive cases:    {df['target'].sum():,}")
print(f"Readmission rate:  {rate:.1%}")
print()
print("Note the class imbalance — this will drive every evaluation decision")
print("in notebook 03. Accuracy will be a useless metric here.")

Final cohort:      99,316 encounters
Positive cases:    11,303
Readmission rate:  11.4%

Note the class imbalance — this will drive every evaluation decision
in notebook 03. Accuracy will be a useless metric here.


---
## 7. Audit trail

The exclusion table below goes straight into your README. It is the first
thing a reviewer should see, because it shows the analysis was designed rather
than assembled.

In [26]:
trail = pd.DataFrame(audit[1:], columns=["Step", "n"])
trail["Removed"] = trail["n"].shift(1).fillna(trail["n"]).astype(int) - trail["n"]
trail["% of raw"] = (trail["n"] / audit[0][1] * 100).round(1)
print(trail.to_string(index=False))

print(f"\nFinal cohort: {len(df):,} encounters x {df.shape[1]} columns")
print("(12 columns dropped in the variable audit: 11 near-constant, 1 for missingness)")

                                      Step      n  Removed  % of raw
                               Raw dataset 101766        0     100.0
                   After removing deceased 100114     1652      98.4
         After removing hospice discharges  99343      771      97.6
After removing encounters that never ended  99319       24      97.6
     After removing unknown/invalid gender  99316        3      97.6

Final cohort: 99,316 encounters x 39 columns
(12 columns dropped in the variable audit: 11 near-constant, 1 for missingness)


In [27]:
import os
os.makedirs(f"{DATA_DIR}/processed", exist_ok=True)
df.to_csv(f"{DATA_DIR}/processed/cohort.csv", index=False)
print(f"Saved cohort: {len(df):,} encounters x {df.shape[1]} columns")

Saved cohort: 99,316 encounters x 39 columns


---
## 8. Summary

**Cohort.** I ended up with 99,316 encounters out of the original 101,766. What got
removed came out for three different reasons, and I think keeping those reasons
separate matters more than the row count does.

Patients who died — in hospital or shortly after discharge — can't be readmitted.
Leaving them in hands the model a perfect predictor, and one it only gets to see at
the exact moment the prediction is supposed to be useful. Hospice patients are
different: they *can* come back, but preventing that isn't the point of their care.
Flagging someone on comfort care for a medication reconciliation and a follow-up
appointment isn't just useless, it's the wrong thing to do. And then there were
encounters that never actually ended — the patient was still admitted, or was
admitted straight back into the same facility. There's no discharge, so there's
nothing for the 30-day clock to start from. Three encounters also went for unusable
gender values, which changed nothing but is worth saying out loud rather than
leaving as a silent step in the trail.

Twelve columns also went, in the variable audit — eleven drugs prescribed to almost
nobody, and weight, which is 96.9% missing. That leaves 38 features plus the target.

Three groups I looked at hard and kept. Transferred patients I nearly excluded,
assuming their readmissions were happening somewhere I couldn't see — then I tested
it, and the numbers said otherwise. Patients who left against medical advice stayed
in because they went home under-treated, and that's exactly the patient I want the
model to notice. The 849 children and adolescents stayed too, even though their
readmission rate is less than half the cohort's, because I'd rather measure how
badly the model handles them than assume it and delete them. Repeat encounters
stayed as well — I deal with the leakage at the split instead of throwing away
nearly a third of the data to get it.

I'm predicting encounters, not patients, because that's how the question gets asked
on a ward. Not "is this person at risk", but "this discharge, today — are they
coming back?"

**Label.** A positive case is someone who walked out of one of these 130 hospitals
alive and was admitted again within 30 days, for any reason. That last part
matters: the data can't tell me whether a return was planned, so a patient coming
back for scheduled treatment looks identical to one coming back in DKA. And the
negative class isn't "did fine" — it's people who never came back mixed in with the
35.7% who came back later. So what I'm really training is a model that spots
*early* returns, which isn't the same thing as spotting patients who do badly. A
discharge team using this should know which of those two it's actually getting.

**Hardest decision.** The 30-day cut-off, and I still don't fully like it.

It's there because Medicare penalises hospitals on that number, not because
anything happens to a patient between day 29 and day 31. People come back for
disease progression, for running out of medication, because nobody at home could
help, because they're old and frail — none of that arranges itself around a 30-day
line. And more than a third of this cohort came back after it, which to me is the
clearest evidence the boundary is drawn in the wrong place.

What I'd actually want is time-to-event: model the hazard continuously and stop
pretending there's a cliff. I can't. The dataset gives me three categories and no
dates, so there's no timing structure to recover. That's a limit of the data, not a
shortcut I took.

I went with 30 days because of what the tool is for. Medication reconciliation, a
diabetes educator review, an appointment booked before the patient leaves, a call
at 72 hours — those act on the first few weeks. So the prediction window lines up
with the window I can actually do something in, even though it doesn't line up with
the biology. That's a reason for this specific tool, not a defence of the cut-off.
If I were building something to decide who needs closer chronic care follow-up,
I'd have answered differently.